In [ ]:
# Setup 1
!apt-get install -y imagemagick-6.q16
!pip install Wand

# Setup 2
!pip install -q win2xcur
!apt-get update -qq
!apt-get install -y -qq imagemagick

from google.colab import files
import zipfile
import os
import subprocess
import glob
import re

# Upload
print("📤 Upload your Xcursor zip:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Extract
!rm -rf work output
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall("work")

# Auto-find cursor folder (rename loop variable to avoid overwriting 'files')
cursor_path = None
for root, dirs, file_list in os.walk("work"):   # <-- 'file_list' instead of 'files'
    if "cursors" in dirs:
        cursor_path = os.path.join(root, "cursors")
        break
    elif any('.' not in f and f.lower() not in ['index.theme', 'readme', 'license', 'copying'] for f in file_list):
        cursor_path = root
        break

if not cursor_path:
    cursor_path = "work"

# Get theme name
theme = os.path.basename(os.path.dirname(cursor_path)) if os.path.dirname(cursor_path) != "work" else "Custom"
for f in glob.glob("work/**/index.theme", recursive=True):
    with open(f, encoding='utf-8', errors='ignore') as file:
        for line in file:
            if line.startswith("Name="):
                theme = line.split("=")[1].strip()
                break

# Sanitize theme for filename
safe_theme = re.sub(r'[^\w\s-]', '', theme).strip().replace(' ', '_')

# Convert
os.makedirs("output", exist_ok=True)
print(f"🔄 Converting {theme}...")
try:
    subprocess.run(['x2wincurtheme', cursor_path, '-n', theme, '-o', 'output'], check=True, capture_output=True)
except:
    print("⚠️ Batch failed, trying individual...")
    for f in os.listdir(cursor_path):
        fp = os.path.join(cursor_path, f)
        if os.path.isfile(fp) and f not in ['index.theme', 'README', 'LICENSE']:
            subprocess.run(['x2wincur', fp, '-o', 'output'], capture_output=True)

# Package if files exist
out_zip = f"{safe_theme}_Windows.zip"
if os.listdir("output"):
    !cd output && zip -r ../{out_zip} .
    if os.path.exists(out_zip):
        print(f"✅ Done! File: {out_zip} ({os.path.getsize(out_zip)//1024} KB)")
        print("📥 Downloading now...")
        files.download(out_zip)   # <-- Now this works
    else:
        print("❌ Zip creation failed – files are in 'output' folder")
else:
    print("❌ No cursors converted – check your zip structure")